<a href="https://colab.research.google.com/github/vencov/FAV_upsidedown/blob/main/TutGTML04/TutGTfeatures_digits_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Auditory model features for machine learning — spoken digit recognition

**Goal.** Use a simple auditory periphery model (gammatone filterbank + inner hair cell stage) as a *feature extractor* for a classifier, and learn what has to be done around the model so that the machine-learning part is correct.

We do **not** change the auditory model here. It is a fixed front end, exactly like an MFCC or a spectrogram. The classifier is the only part that is trained.

**Pipeline**

`waveform → level scaling → gammatone + IHC → frame averaging → fixed length → normalization → classifier`

**How to work.** Cells marked **TODO** contain `raise NotImplementedError` — replace it with your code. The cell right below each TODO has tests that tell you whether it works. Cells marked *given* should not need editing. Answer the **Questions** in the markdown cells (double-click to edit).

**Data.** [Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) (FSDD): 3000 recordings of the digits 0–9, 6 speakers × 50 repetitions × 10 digits, 8 kHz. Licensed CC BY-SA 4.0.

## 1. Setup and data *(given)*

In [ ]:
import os, glob, subprocess
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram

if not os.path.exists('free-spoken-digit-dataset'):
    subprocess.run(['git', 'clone', '--depth', '1', '-q',
                    'https://github.com/Jakobovski/free-spoken-digit-dataset'], check=True)

files = sorted(glob.glob('free-spoken-digit-dataset/recordings/*.wav'))
digits = np.array([int(os.path.basename(f).split('_')[0]) for f in files])
speakers = np.array([os.path.basename(f).split('_')[1] for f in files])
waves = [wavfile.read(f)[1].astype(float) / 32768 for f in files]
FS = 8000

print(len(files), 'recordings; speakers:', sorted(set(speakers.tolist())))
lens = np.array([len(w) for w in waves]) / FS
print(f'duration: median {np.median(lens):.2f} s, min {lens.min():.2f} s, max {lens.max():.2f} s')

## 2. The auditory model *(given)*

Your gammatone filterbank with impulse-invariant coefficients, followed by a simple IHC stage (half-wave rectification → lowpass → power-law compression).

Note the output convention of `fce_BMgammatone`: array `[n_samples, n_channels]` with the **highest CF in column 0**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter, butter
import matplotlib.ticker as mticker

# ---------------- Gammatone filterbank (gammaf1.m / getGFBFilterCoefs.m) ----------------
def gammaf1(fs, f0, beta, n, ftype):
    """Digital gammatone filter (n-th order) as a complex-coefficient IIR.
    ftype = 1: bilinear transform, 2: impulse invariance."""
    if ftype == 1:
        w0 = 2 * np.pi * f0 / fs
        W0 = np.tan(w0 / 2)
        Beta = np.tan(np.pi * beta / fs)
        tmp = 1 + 1 / Beta - 1j * W0 / Beta
        b = np.array([1, 1], dtype=complex) / tmp
        a = np.array([tmp, 1 - 1 / Beta - 1j * W0 / Beta]) / tmp
    elif ftype == 2:
        b = np.array([1 - np.exp(-2 * np.pi * beta / fs)], dtype=complex)
        a = np.array([1, -np.exp(-(2 * np.pi * beta + 1j * 2 * np.pi * f0) / fs)])
    else:
        raise ValueError("ftype must be 1 or 2")
    b2, a2 = np.array([1.0 + 0j]), np.array([1.0 + 0j])
    for _ in range(n):                      # cascade n identical 1st-order sections
        b2 = np.convolve(b, b2)
        a2 = np.convolve(a, a2)
    return b2, a2


def getGFBFilterCoefs(fs, centerfreq, bw):
    centerfreq = np.atleast_1d(centerfreq)
    beta = 1.0183 * np.atleast_1d(bw)       # 4th-order gammatone (0.637 would be 2nd order)
    b, a = [], []
    for k in range(len(centerfreq)):
        bt, at = gammaf1(fs, centerfreq[k], beta[k], 4, 2)
        b.append(bt); a.append(at)
    return np.vstack(b), np.vstack(a)


class BMParams:
    """Container replacing the MATLAB struct CBM."""
    pass


def initBMGTpar(cfArray, fs):
    cfArray = np.asarray(cfArray, dtype=float)
    bandwidth = 24.7 + cfArray / 9.265      # ERB (Glasberg & Moore)
    CBM = BMParams()
    CBM.GT_b, CBM.GT_a = getGFBFilterCoefs(fs, cfArray, bandwidth)
    CBM.bandwidth = bandwidth
    CBM.cfArray = cfArray
    return CBM


def fce_BMgammatone(signal, par, pastGTn=None):
    """Filter `signal` through the gammatone filterbank.
    Returns out (n_samples x n_channels, HIGH CF in column 0) and filter states."""
    signal = np.asarray(signal).ravel()
    nsect = len(par.cfArray)
    out = np.zeros((len(signal), nsect))
    order = par.GT_a.shape[1] - 1
    if pastGTn is None or np.size(pastGTn) == 0:
        pastGTn = np.zeros((order, nsect), dtype=complex)
    for ch in range(nsect):
        outBM, pastGTn[:, ch] = lfilter(par.GT_b[ch], par.GT_a[ch], signal, zi=pastGTn[:, ch])
        out[:, ch] = 2 * np.real(outBM)
    return out[:, ::-1], pastGTn            # fliplr: highest freq -> lowest-index section


def erb_space(f_low, f_high, n):
    """n center frequencies spaced uniformly on the ERB scale (Moore & Glasberg 1983)."""
    ear_q, min_bw = 9.26449, 24.7
    idx = np.arange(n, 0, -1)
    cf = -(ear_q * min_bw) + np.exp(
        idx * (-np.log(f_high + ear_q * min_bw) + np.log(f_low + ear_q * min_bw)) / n
    ) * (f_high + ear_q * min_bw)
    return np.sort(cf)


# ---------------------------------- Simple IHC model -------------------------------------
def simple_ihc(bm_out, fs, lp_cutoff=1000, lp_order=2, compression_exp=0.3):
    """Half-wave rectification -> lowpass (loss of phase-locking) -> compressive power-law."""
    rect = np.maximum(bm_out, 0.0)
    b, a = butter(lp_order, lp_cutoff / (fs / 2))
    filtered = lfilter(b, a, rect, axis=0)
    filtered = np.maximum(filtered, 0.0)    # clip ringing before the fractional power
    return filtered ** compression_exp

# ---------------------------------- Level scaling -------------------------------------

def scale_to_rms(x, rms=0.05):
    """Scale waveform x so that its root-mean-square value equals `rms`."""
    return x * rms / np.sqrt(np.mean(x ** 2))


We use 32 channels, ERB-spaced between 100 and 3400 Hz. The recordings are sampled at 8 kHz, so there is no information above 4 kHz and no point in placing channels there.

In [ ]:
cfs = erb_space(100, 3400, 32)
par = initBMGTpar(cfs, FS)

def auditory_model(x, lp_cutoff=1000):
    """waveform -> IHC output, shape [n_samples, n_channels] (high CF first)"""
    bm, _ = fce_BMgammatone(x, par)
    return simple_ihc(bm, FS, lp_cutoff=lp_cutoff)

print('CFs (Hz):', np.round(cfs[[0, 8, 16, 24, 31]]))

## 3. First look at the model output *(given)*

Compare the model output with an ordinary spectrogram of the same digit.

In [ ]:
from IPython.display import Audio, display

def compare_speakers(digit, spk_a, spk_b, rep=0, rms=0.05):
    """Same digit, two speakers: play the sounds and plot waveform, model output and spectrogram."""
    idx = [np.where((digits == digit) & (speakers == s))[0][rep] for s in (spk_a, spk_b)]
    fig, ax = plt.subplots(3, 2, figsize=(11, 7), sharex='col')
    for col, i in enumerate(idx):
        x = waves[i]
        t = np.arange(len(x)) / FS
        r = auditory_model(scale_to_rms(x))
        print(f'{speakers[i]}:')
        display(Audio(x, rate=FS))
        ax[0, col].plot(t, x, lw=0.5)
        ax[0, col].set_title(f'digit {digits[i]}, {speakers[i]}')
        ax[1, col].imshow(r.T, aspect='auto', cmap='magma', extent=[0, t[-1], r.shape[1], 0])
        f, tt, S = spectrogram(x, FS, nperseg=160, noverlap=80)
        ax[2, col].pcolormesh(tt, f, 10 * np.log10(S + 1e-12), shading='auto', cmap='magma')
        ax[2, col].set_xlabel('time (s)')
    ax[0, 0].set_ylabel('amplitude')
    ax[1, 0].set_ylabel('channel (0 = high CF)')
    ax[2, 0].set_ylabel('Hz')
    plt.tight_layout(); plt.show()

compare_speakers(digit=3, spk_a='george', spk_b='theo')

## 4. Feature extraction

### 4.1 Level scaling *(given)*

The recordings have an arbitrary gain, which depends on the microphone and the recording setup, and the output of the auditory model depends on the input level (the compressive power law in the IHC stage means that the output does not change in proportion to the input). To make the clips comparable, every clip is scaled to the same RMS value before it enters the model. The model has no absolute reference (no SPL, no threshold), so the value 0.05 is an arbitrary convention. We return to the effect of level in the last task.

### 4.2 Frame averaging — TODO

The IHC output has 8000 samples per second, far more than needed. Average it in windows of `win` samples, moving by `hop` samples (here 20 ms window = 160 samples, 10 ms hop = 80 samples). “numpy.lib.stride_tricks.sliding_window_view can help, but a loop over frames is also acceptable.

Input `r`: `[n_samples, n_channels]`. Output: `[n_frames, n_channels]` with `n_frames = (n_samples - win) // hop + 1` (only complete windows).

In [ ]:
HOP, WIN = 80, 160

HOP, WIN = 80, 160   # 10 ms hop and 20 ms window at fs = 8 kHz

def frame_average(r, hop=HOP, win=WIN):
    """Mean of r over windows of `win` samples, shifted by `hop` samples.

    r: array [n_samples, n_channels]
    returns: array [n_frames, n_channels], n_frames = (n_samples - win) // hop + 1
    """
    n_frames = (r.shape[0] - win) // hop + 1
    # TODO
    # Hint: a loop over the frames is fine; alternatively
    # numpy.lib.stride_tricks.sliding_window_view(r, win, axis=0) gives all windows at once.
    raise NotImplementedError


In [ ]:
# test
r_test = np.arange(10, dtype=float)[:, None] * np.ones((1, 3))

print(r_test)

out = frame_average(r_test, hop=2, win=4)

print(out)
assert out.shape == (4, 3), out.shape
assert np.allclose(out[:, 0], [1.5, 3.5, 5.5, 7.5])
print('OK')

### 4.3 Fixed length — TODO

Clips have different durations (see the numbers above), but a classifier needs inputs of equal size. Two common options:

- **pad / crop** to a fixed number of frames (provided below),
- **time-normalize**: stretch or squeeze every utterance to the same number of frames by interpolation along time (you write this one).

Implement `time_normalize(f, n)` with `numpy.interp`, applied independently to each channel. Input `f`: `[n_frames, n_channels]`, output `[n, n_channels]`.

In [ ]:
def pad_or_crop(f, n):
    """(given) Zero-pad or cut the feature map to exactly n frames."""
    if f.shape[0] >= n:
        return f[:n]
    return np.vstack([f, np.zeros((n - f.shape[0], f.shape[1]))])

def time_normalize(f, n):
    """Resample the feature map f to exactly n frames by linear interpolation along time.

    f: array [n_frames, n_channels]
    returns: array [n, n_channels]; the first and the last frame are preserved.
    """
    xi = np.linspace(0, f.shape[0] - 1, n)
    # TODO
    # Hint: for every channel, evaluate the interpolated curve at n equally spaced
    # positions between frame 0 and the last frame, using numpy.interp.
    raise NotImplementedError


In [ ]:
# test
f_test = np.random.rand(37, 5)
g = time_normalize(f_test, 20)
assert g.shape == (20, 5)
assert np.allclose(g[0], f_test[0]) and np.allclose(g[-1], f_test[-1])
assert np.allclose(time_normalize(np.ones((11, 2)) * 3, 7), 3)
print('OK')

### 4.4 Put it together *(given)*

`make_features` runs the whole chain for a list of waveforms. The options (`rms`, `lp_cutoff`, `snr_db`, `n_frames`) are used in the simulations later.

In [ ]:
N_FRAMES = 20

def make_features(wavs, rms=0.05, lp_cutoff=1000, snr_db=None, n_frames=N_FRAMES, seed=0):
    """waveforms -> array [n_clips, n_frames, n_channels]"""
    rng = np.random.default_rng(seed)
    feats = []
    for x in wavs:
        x = scale_to_rms(x, rms)
        if snr_db is not None:                       # white noise at given overall SNR
            x = x + rng.standard_normal(len(x)) * rms * 10 ** (-snr_db / 20)
        r = auditory_model(x, lp_cutoff=lp_cutoff)
        feats.append(time_normalize(frame_average(r), n_frames))
    return np.array(feats)

import time
t0 = time.time()
F = make_features(waves)
print(F.shape, f'({time.time() - t0:.0f} s)')

Look at the three feature maps below (same digit, three speakers). Notice what is similar, what differs and why can a classifier not simply compare maps of different speakers pixel by pixel.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for a, spk in zip(ax, ['george', 'jackson', 'theo']):
    k = np.where((digits == 3) & (speakers == spk))[0][0]
    a.imshow(F[k].T, aspect='auto', cmap='magma'); a.set_title(f'digit 3, {spk}'); a.set_xlabel('frame')
ax[0].set_ylabel('channel (0 = high CF)'); plt.tight_layout(); plt.show()

## 5. Train / test split and normalization

### 5.1 Split by speaker *(given)*

Create boolean masks `train_mask`, `test_mask` over all 3000 clips so that **no speaker appears in both sets**: test on speakers `theo` and `yweweler`, train on the other four.

In [ ]:
test_speakers = ['theo', 'yweweler']

test_mask = np.isin(speakers, test_speakers)
train_mask = ~test_mask

In [ ]:
# test
assert train_mask.dtype == bool and test_mask.dtype == bool
assert not (train_mask & test_mask).any()
assert not set(speakers[train_mask]) & set(speakers[test_mask])
print(train_mask.sum(), 'train clips,', test_mask.sum(), 'test clips')

### 5.2 Normalization — TODO

Neural networks and logistic regression work much better when every input feature has zero mean and unit variance. Write `normalize(X_train, X_test)`, which (1) for each of the clips flattens the sample x channel feature matrix into a vector, standardizes **each feature** with the mean and standard deviation computed from the **training set only**, and returns both sets.

Notice that the test set is not used to compute the mean and standard deviation.


In [ ]:
def normalize(X_train, X_test):
    """Flatten the clips to vectors and standardize each feature using training-set statistics.

    X_train: array [n_train, n_frames, n_channels]
    X_test:  array [n_test,  n_frames, n_channels]
    returns: (A, B), both of shape [n_clips, n_frames * n_channels]; every column of A has
             mean 0 and standard deviation 1, and B is transformed with the SAME numbers.
    """
    # TODO
    # Hint: reshape(len(X), -1) turns every feature map into one row vector.
    # Add a small constant (1e-8) to the standard deviation to avoid division by zero.
    raise NotImplementedError


In [ ]:
# test
A, B = normalize(F[train_mask], F[test_mask])
assert A.shape == (train_mask.sum(), N_FRAMES * len(cfs))
assert np.allclose(A.mean(0), 0, atol=1e-6) and np.allclose(A.std(0), 1, atol=1e-3)
assert not np.allclose(B.mean(0), 0, atol=1e-6)        # test set is NOT exactly standardized
print('OK')

## 6. Classifier *(given)*

Multinomial logistic regression on the flattened feature map (20 frames × 32 channels = 640 inputs).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay

def evaluate(F_train, y_train, F_test, y_test, return_model=False):
    A, B = normalize(F_train, F_test)
    clf = LogisticRegression(max_iter=3000, C=0.1).fit(A, y_train)
    acc = (clf.predict(B) == y_test).mean()
    return (acc, clf, B) if return_model else acc

acc, clf, B = evaluate(F[train_mask], digits[train_mask], F[test_mask], digits[test_mask], return_model=True)
print(f'Speaker-independent test accuracy: {acc:.3f}   (chance = 0.100)')
ConfusionMatrixDisplay.from_predictions(digits[test_mask], clf.predict(B), cmap='Blues'); plt.show()

Notice which digits are confused most often.

### 6.1 Why we split by speaker

Now repeat the experiment with a **random** split of the clips (same sizes), ignoring who is speaking.

In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(F))
rand_train = np.zeros(len(F), bool); rand_train[perm[:train_mask.sum()]] = True

acc_rand = evaluate(F[rand_train], digits[rand_train], F[~rand_train], digits[~rand_train])
print(f'random split: {acc_rand:.3f}    speaker split: {acc:.3f}')


## 7. Reference: a plain spectrogram *(given)*

To judge whether the auditory model helps, we need a baseline with the same classifier and the same time handling. `spec_features` computes log energy in the same 32 ERB-spaced bands directly from a spectrogram (no gammatone filters, no IHC stage).

In [ ]:
def ebin(f):                                     # ERB-number scale
    return 9.26449 * np.log(1 + f / (9.26449 * 24.7))

def spec_features(wavs, rms=0.05, snr_db=None, n_frames=N_FRAMES, seed=0):
    rng = np.random.default_rng(seed)
    ec = ebin(cfs)
    edges = np.concatenate([[-np.inf], (ec[1:] + ec[:-1]) / 2, [np.inf]])
    feats = []
    for x in wavs:
        x = scale_to_rms(x, rms)
        if snr_db is not None:
            x = x + rng.standard_normal(len(x)) * rms * 10 ** (-snr_db / 20)
        f, _, S = spectrogram(x, FS, nperseg=WIN, noverlap=WIN - HOP, window='hann')
        e = ebin(f)
        B = np.stack([S[(e >= edges[i]) & (e < edges[i + 1])].sum(0) for i in range(len(cfs))], axis=1)
        B = np.maximum(B, B.max() * 1e-8)            # floor 80 dB below the loudest value (relative, so level-independent)
        feats.append(time_normalize(np.log(B), n_frames))
    return np.array(feats)

G = spec_features(waves)
acc_spec = evaluate(G[train_mask], digits[train_mask], G[test_mask], digits[test_mask])
print(f'spectrogram baseline: {acc_spec:.3f}     auditory model: {acc:.3f}')

## 8. Furher simlations and validation

For each experiment fill in the table and describe what you see. The helper functions are given; **train on clean speech at the default settings** (`F`, `G` from above) and change only the test condition, unless said otherwise. Notice how to performance of the classifier in noisy condition compares to human performance which allows for 50% detections at SNR of about -5 to -10 dB.

### 8.1 Noise robustness

In [ ]:
snrs = [None, 20, 10, 5, 0]
res_aud, res_spec = [], []
for snr in snrs:
    Ft = make_features([waves[k] for k in np.where(test_mask)[0]], snr_db=snr)
    Gt = spec_features([waves[k] for k in np.where(test_mask)[0]], snr_db=snr)
    res_aud.append(evaluate(F[train_mask], digits[train_mask], Ft, digits[test_mask]))
    res_spec.append(evaluate(G[train_mask], digits[train_mask], Gt, digits[test_mask]))

plt.plot(range(len(snrs)), res_aud, 'o-', label='auditory model')
plt.plot(range(len(snrs)), res_spec, 's-', label='spectrogram')
plt.xticks(range(len(snrs)), ['clean', '20 dB', '10 dB', '5 dB', '0 dB']); plt.axhline(0.1, color='gray', ls=':')
plt.xlabel('SNR of test clips (white noise)'); plt.ylabel('accuracy'); plt.legend(); plt.show()

### 8.2 Level mismatch

Train at the default level (RMS 0.05) and test at levels 20 dB lower and 20 dB higher (RMS 0.005 and 0.5).

In [ ]:
for name, rms in [('-20 dB', 0.005), ('default', 0.05), ('+20 dB', 0.5)]:
    Ft = make_features([waves[k] for k in np.where(test_mask)[0]], rms=rms)
    Gt = spec_features([waves[k] for k in np.where(test_mask)[0]], rms=rms)
    print(f'{name:8s} auditory: {evaluate(F[train_mask], digits[train_mask], Ft, digits[test_mask]):.3f}'
          f'    spectrogram: {evaluate(G[train_mask], digits[train_mask], Gt, digits[test_mask]):.3f}')


### 8.3 IHC lowpass cutoff

The IHC lowpass models the loss of phase-locking at higher frequencies. Rerun feature extraction (training and testing!) with `lp_cutoff` = 300, 1000, 3000 Hz.

In [ ]:
for lp in [300, 1000, 3000]:
    Fl = make_features(waves, lp_cutoff=lp)
    print(lp, 'Hz:', round(evaluate(Fl[train_mask], digits[train_mask], Fl[test_mask], digits[test_mask]), 3))

### 8.4 Time handling

Replace `time_normalize` by `pad_or_crop` (70 frames) in `make_features` and rerun the baseline. Compare the accuracies and explain which step of the processing is responsible for the difference.

### 8.5 Possible extensions

A small neural network (MLP, CNN) instead of logistic regression; leave-one-speaker-out cross-validation; simulated hearing loss in the front end (raised thresholds or broader filters) with a classifier trained on healthy features.